# Technology Diffusion Approximation Validation

This notebook compares, on many small preferential-attachment instances:

- the repository's connected **Approximation** algorithm;
- the **Goldberg–Liu** simple permutation IP, whose optimum is denoted $|OPT_{connected}|$ because it restricts attention to connected activation sequences;
- the unrestricted **exact optimum** $|OPT|$, found by brute-force subset enumeration; and
- the theoretical upper bound

$$
|OPT_{connected}| \leq \min\{2|OPT|,\ |OPT|+\ell\},
$$

where $\ell$ is the number of distinct threshold values actually present in the instance.

Here “connected” describes the activation sequence used by the Goldberg–Liu simple IP; it does not mean that the initial seed set's induced subgraph must be connected. The plotted Approximation and Goldberg–Liu colors are copied from `notebooks/results_try.ipynb`: blue `#1f77b4` and green `#2ca02c`.

## Imports and repository setup

The package uses the standard importable path `src/technology_diffusion`. The setup below adds the repository's `src` directory when the notebook is run from either the repository root or `notebooks/`.

In [ ]:
from __future__ import annotations

import itertools
import sys
import time
from pathlib import Path

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/ directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from technology_diffusion import (
    approx,
    build_golberg_liu_ip,
    connected_component_spread,
    create_pa_graph,
    get_activation_sequence_and_seeds,
    make_vector,
)

pd.set_option("display.max_columns", 100)
print(f"Project root: {PROJECT_ROOT}")


## Experiment configuration

The defaults generate $5 \times 4 \times 8 = 160$ instances. Values of $n$ are deliberately small because exact subset enumeration is exponential. Increase them cautiously.

`ell` is computed as `len(set(thetas.values()))`, not from the number of values that the generator could have sampled: the theorem depends on threshold values realized by the instance.

In [ ]:
N_VALUES = [6, 8, 10, 12, 14]
C_VALUES = [1, 2, 3, 4]
RANDOM_SEEDS = list(range(1, 9))
INIT_NODES = 5
INIT_MODE = "complete"
SAVE_OUTPUTS = True

RESULTS_DIR = PROJECT_ROOT / "results" / "technology diffusion"
FIGURES_DIR = PROJECT_ROOT / "figures" / "technology_diffusion_approximation_validation"
RESULTS_PATH = RESULTS_DIR / "technology_diffusion_approximation_validation_results.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.titlesize": 17,
})

# Approximation and Goldberg–Liu exactly match results_try.ipynb.
ALGO_COLORS = {
    "approx": "#1f77b4",
    "goldberg_liu": "#2ca02c",
    "exact_connected": "#d62728",
    "theory_bound": "#111111",
}
ALGO_LINESTYLES = {
    "approx": "--",
    "goldberg_liu": "-.",
    "exact_connected": "-",
    "theory_bound": ":",
}

print(f"Instances: {len(N_VALUES) * len(C_VALUES) * len(RANDOM_SEEDS)}")
print(f"CSV output: {RESULTS_PATH}")

## Exact brute force and the source Goldberg–Liu implementation

`brute_force_optimum` checks all seed subsets in increasing cardinality, so the first feasible set is the unrestricted exact optimum $OPT$.

Goldberg–Liu is run through `build_golberg_liu_ip` and `get_activation_sequence_and_seeds` imported from `technology_diffusion.ip_problems` through the package API. In particular, this notebook preserves the source objective's `t <= theta[v]` convention even though `connected_component_spread` uses `theta[v] - 1`. No offset or corrected local IP is applied. Any empirical discrepancy caused by that convention remains visible in the results and bound-check table.

In [ ]:
def activates_all(g: nx.Graph, thetas: dict[int, int], seeds: tuple[int, ...] | set[int]) -> bool:
    n_nodes = g.number_of_nodes()
    x0 = make_vector(seeds, n_nodes)
    spread, _, _ = connected_component_spread(g, x0, thetas)
    return spread == n_nodes


def brute_force_optimum(
    g: nx.Graph,
    thetas: dict[int, int],
) -> tuple[int, tuple[int, ...], int, float]:
    """Return unrestricted (optimal size, one optimal set, subsets tested, runtime)."""
    start = time.perf_counter()
    nodes = tuple(sorted(g.nodes()))
    if nodes != tuple(range(len(nodes))):
        raise ValueError("The repository evaluator expects nodes labelled 0, ..., n-1.")

    evaluated = 0
    for k in range(1, len(nodes) + 1):
        for seed_tuple in itertools.combinations(nodes, k):
            evaluated += 1
            if activates_all(g, thetas, seed_tuple):
                return k, seed_tuple, evaluated, time.perf_counter() - start

    raise RuntimeError("No feasible seed set was found.")


def solve_goldberg_liu_source(
    g: nx.Graph,
    thetas: dict[int, int],
) -> tuple[int, tuple[int, ...], tuple[int, ...], float]:
    """Build and solve the repository's Goldberg–Liu IP without changing its thresholds."""
    start = time.perf_counter()
    model, xvars = build_golberg_liu_ip(g, thetas)
    if model is None or xvars is None:
        raise RuntimeError("The source Goldberg–Liu model could not be built.")

    model.Params.OutputFlag = 0
    model.optimize()
    if model.Status != GRB.OPTIMAL:
        status = model.Status
        model.dispose()
        raise RuntimeError(f"Goldberg–Liu did not solve to optimality (status={status}).")

    _, order, seeds = get_activation_sequence_and_seeds(g, thetas, xvars)
    objective = int(round(model.ObjVal))
    runtime = time.perf_counter() - start
    model.dispose()

    seeds = tuple(sorted(int(u) for u in seeds))
    order = tuple(int(u) for u in order)
    if objective != len(seeds):
        raise AssertionError("Goldberg–Liu objective and source-extracted seed set disagree.")
    if not activates_all(g, thetas, seeds):
        raise AssertionError("The source Goldberg–Liu seed set is not feasible under the evaluator.")
    return objective, seeds, order, runtime

## Run all instances

Approximation failures are retained as missing values in `approx_k`. For visualization only, a failed run is replaced by the always-feasible solution containing all $n$ vertices (`approx_k_for_plot = n`). This convention is explicit in every affected plot, and `approx_success` is reported separately.

In [ ]:
def run_instance(n_nodes: int, c: int, seed: int) -> dict:
    g, thetas = create_pa_graph(
        n_nodes=n_nodes,
        c=c,
        seed=seed,
        init_nodes=INIT_NODES,
        init_mode=INIT_MODE,
    )
    threshold_values = tuple(sorted(set(int(v) for v in thetas.values())))
    ell = len(threshold_values)

    opt_k, opt_seeds, opt_tests, opt_runtime = brute_force_optimum(g, thetas)
    gl_k, gl_seeds, gl_order, gl_runtime = solve_goldberg_liu_source(g, thetas)

    approx_k, approx_x, approx_runtime, _ = approx(g, thetas)
    approx_success = approx_k is not None and approx_x is not None
    approx_seeds = tuple(np.flatnonzero(approx_x).astype(int)) if approx_success else tuple()
    approx_feasible = (
        activates_all(g, thetas, approx_seeds)
        and (len(approx_seeds) <= 1 or nx.is_connected(g.subgraph(approx_seeds)))
        if approx_success
        else False
    )

    # The requested bound uses unrestricted OPT from brute force.
    bound_2opt = 2 * opt_k
    bound_opt_plus_ell = opt_k + ell
    theory_bound = min(bound_2opt, bound_opt_plus_ell)

    return {
        "n_nodes": n_nodes,
        "c": c,
        "seed": seed,
        "n_edges": g.number_of_edges(),
        "threshold_values": threshold_values,
        "ell": ell,
        "opt_bruteforce": opt_k,
        "approx_k": int(approx_k) if approx_success else np.nan,
        "goldberg_liu_k": gl_k,
        "bound_2opt": bound_2opt,
        "bound_opt_plus_ell": bound_opt_plus_ell,
        "theory_bound": theory_bound,
        "active_bound_branch": "2OPT" if bound_2opt <= bound_opt_plus_ell else "OPT+ell",
        "opt_seeds": opt_seeds,
        "approx_seeds": approx_seeds,
        "goldberg_liu_seeds": gl_seeds,
        "goldberg_liu_order": gl_order,
        "opt_bf_subsets_tested": opt_tests,
        "opt_bf_runtime_s": opt_runtime,
        "approx_runtime_s": approx_runtime,
        "goldberg_liu_runtime_s": gl_runtime,
        "approx_success": approx_success,
        "approx_feasible": approx_feasible,
        "goldberg_is_upper_bound": gl_k >= opt_k,
        "bound_holds_for_goldberg": gl_k <= theory_bound,
    }


combinations = list(itertools.product(N_VALUES, C_VALUES, RANDOM_SEEDS))
rows = []
start_all = time.perf_counter()
for run_idx, (n_nodes, c, seed) in enumerate(combinations, start=1):
    rows.append(run_instance(n_nodes, c, seed))
    if run_idx == 1 or run_idx % 10 == 0 or run_idx == len(combinations):
        print(f"Completed {run_idx:>3}/{len(combinations)} instances")

df = pd.DataFrame(rows)
df["approx_k_for_plot"] = df["approx_k"].fillna(df["n_nodes"])
df["goldberg_to_opt_ratio"] = df["goldberg_liu_k"] / df["opt_bruteforce"]
df["approx_to_opt_ratio"] = df["approx_k_for_plot"] / df["opt_bruteforce"]
df["bound_to_opt_ratio"] = df["theory_bound"] / df["opt_bruteforce"]
df["bound_slack_for_goldberg"] = df["theory_bound"] - df["goldberg_liu_k"]

if SAVE_OUTPUTS:
    df.to_csv(RESULTS_PATH, index=False)

print(f"Finished in {time.perf_counter() - start_all:.2f}s")
display(df.head(10))

## Correctness checks and compact summary

Goldberg–Liu and successful Approximation outputs must be feasible, so they cannot beat the brute-force optimum. The requested bound is reported empirically rather than asserted: this keeps the source `t <= theta` implementation visible even if its threshold convention produces a discrepancy with the `theta - 1` diffusion evaluator.

In [ ]:
assert df["goldberg_is_upper_bound"].all(), "A feasible Goldberg–Liu result beats exact OPT."
assert df.loc[df["approx_success"], "approx_feasible"].all(), "A reported approximation is infeasible."

summary_by_c_n = (
    df.groupby(["c", "n_nodes"], as_index=False)
      .agg(
          instances=("seed", "size"),
          mean_ell=("ell", "mean"),
          mean_opt=("opt_bruteforce", "mean"),
          mean_goldberg=("goldberg_liu_k", "mean"),
          mean_approx=("approx_k_for_plot", "mean"),
          mean_bound=("theory_bound", "mean"),
          approx_success_rate=("approx_success", "mean"),
          bound_holds_rate=("bound_holds_for_goldberg", "mean"),
          bound_tight_rate=("bound_slack_for_goldberg", lambda x: float((x == 0).mean())),
      )
)

print(f"Goldberg–Liu is at least exact OPT:     {df['goldberg_is_upper_bound'].mean():.0%}")
print(f"Requested theoretical bound holds:     {df['bound_holds_for_goldberg'].mean():.1%}")
print(f"Approximation success rate:            {df['approx_success'].mean():.1%}")
print(f"Bound is tight for Goldberg–Liu:       {(df['bound_slack_for_goldberg'] == 0).mean():.1%}")
print("Active bound branches:")
display(df["active_bound_branch"].value_counts().rename_axis("branch").to_frame("instances"))
display(summary_by_c_n)

## Absolute solution sizes vs. the theoretical bound

Each panel fixes $c$ and averages over random seeds. Red is exact unrestricted $OPT$ from brute force; green is the source Goldberg–Liu result; and the black dotted line is $\min\{2|OPT|, |OPT|+\ell\}$. Failed Approximation runs are plotted at the feasible fallback $n$ and quantified in the later success-rate heatmap.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=130, sharex=True, sharey=True)
for ax, c in zip(axes.flat, C_VALUES):
    part = summary_by_c_n[summary_by_c_n["c"] == c].sort_values("n_nodes")
    x = part["n_nodes"]
    ax.plot(
        x, part["mean_approx"], marker="o", linewidth=2,
        color=ALGO_COLORS["approx"], linestyle=ALGO_LINESTYLES["approx"],
        label="Approximation (failure → n)",
    )
    ax.plot(
        x, part["mean_goldberg"], marker="s", linewidth=2,
        color=ALGO_COLORS["goldberg_liu"], linestyle=ALGO_LINESTYLES["goldberg_liu"],
        label=r"Goldberg–Liu ($OPT_{connected}$)",
    )
    ax.plot(
        x, part["mean_opt"], marker="^", linewidth=2,
        color=ALGO_COLORS["exact_connected"], linestyle=ALGO_LINESTYLES["exact_connected"],
        label=r"Exact brute-force $OPT$",
    )
    ax.plot(
        x, part["mean_bound"], marker="D", linewidth=2,
        color=ALGO_COLORS["theory_bound"], linestyle=ALGO_LINESTYLES["theory_bound"],
        label=r"$\min\{2OPT,OPT+\ell\}$",
    )
    ax.set_title(f"c = {c}")
    ax.set_xticks(N_VALUES)
    ax.set_xlabel("Number of nodes n")
    ax.set_ylabel("Mean seed-set size")
    ax.spines[["top", "right"]].set_visible(False)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncols=4, frameon=True, bbox_to_anchor=(0.5, 0.005))
fig.suptitle("Approximation and source Goldberg–Liu vs. exact brute force and theory", y=0.995)
fig.tight_layout(rect=(0, 0.06, 1, 0.97))
if SAVE_OUTPUTS:
    fig.savefig(FIGURES_DIR / "mean_solution_sizes_vs_bound.png", dpi=220, bbox_inches="tight")
plt.show()

## Ratios relative to exact brute-force OPT

The red line at 1 is the exact unrestricted optimum. Goldberg–Liu and the theoretical bound should lie at or above it. Approximation failures again use the feasible fallback $n$.

In [ ]:
ratio_summary = (
    df.groupby(["c", "n_nodes"], as_index=False)
      .agg(
          goldberg_ratio=("goldberg_to_opt_ratio", "mean"),
          approx_ratio=("approx_to_opt_ratio", "mean"),
          bound_ratio=("bound_to_opt_ratio", "mean"),
      )
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=130, sharex=True, sharey=True)
for ax, c in zip(axes.flat, C_VALUES):
    part = ratio_summary[ratio_summary["c"] == c].sort_values("n_nodes")
    x = part["n_nodes"]
    ax.axhline(1, color=ALGO_COLORS["exact_connected"], linewidth=2, label=r"Exact brute-force $OPT$")
    ax.plot(
        x, part["approx_ratio"], marker="o", linewidth=2,
        color=ALGO_COLORS["approx"], linestyle=ALGO_LINESTYLES["approx"],
        label="Approximation / OPT (failure → n)",
    )
    ax.plot(
        x, part["goldberg_ratio"], marker="s", linewidth=2,
        color=ALGO_COLORS["goldberg_liu"], linestyle=ALGO_LINESTYLES["goldberg_liu"],
        label="Goldberg–Liu / OPT",
    )
    ax.plot(
        x, part["bound_ratio"], marker="D", linewidth=2,
        color=ALGO_COLORS["theory_bound"], linestyle=ALGO_LINESTYLES["theory_bound"],
        label="Theory bound / OPT",
    )
    ax.set_title(f"c = {c}")
    ax.set_xticks(N_VALUES)
    ax.set_xlabel("Number of nodes n")
    ax.set_ylabel(r"Mean ratio to exact $OPT$")
    ax.spines[["top", "right"]].set_visible(False)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncols=4, frameon=True, bbox_to_anchor=(0.5, 0.005))
fig.suptitle("Normalized comparison with exact brute-force OPT", y=0.995)
fig.tight_layout(rect=(0, 0.06, 1, 0.97))
if SAVE_OUTPUTS:
    fig.savefig(FIGURES_DIR / "ratios_to_exact_opt.png", dpi=220, bbox_inches="tight")
plt.show()

## Bound usage and approximation success

The left heatmap shows the source Goldberg–Liu value divided by $\min\{2|OPT|,|OPT|+\ell\}$. Values at 1 mean the bound is tight; values above 1 flag an empirical violation under the source threshold convention. The right heatmap makes Approximation failures visible instead of silently dropping them from averages.

In [ ]:
def annotate_heatmap(ax, matrix: np.ndarray, fmt: str, threshold: float) -> None:
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix[row, col]
            color = "white" if value < threshold else "black"
            ax.text(col, row, format(value, fmt), ha="center", va="center", color=color, fontsize=10)


bound_usage = (
    df.assign(usage=df["goldberg_liu_k"] / df["theory_bound"])
      .pivot_table(index="c", columns="n_nodes", values="usage", aggfunc="mean")
      .reindex(index=C_VALUES, columns=N_VALUES)
)
approx_success = (
    df.pivot_table(index="c", columns="n_nodes", values="approx_success", aggfunc="mean")
      .reindex(index=C_VALUES, columns=N_VALUES)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), dpi=130)

bound_vmax = max(1.0, float(np.nanmax(bound_usage.to_numpy())))
im0 = axes[0].imshow(bound_usage.to_numpy(), cmap="YlOrRd", vmin=0, vmax=bound_vmax, aspect="auto")
annotate_heatmap(axes[0], bound_usage.to_numpy(), ".2f", threshold=0.62 * bound_vmax)
axes[0].set_title(r"Bound usage: Goldberg–Liu / bound")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(approx_success.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
annotate_heatmap(axes[1], approx_success.to_numpy(), ".0%", threshold=0.58)
axes[1].set_title("Approximation success rate")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

for ax in axes:
    ax.set_xticks(range(len(N_VALUES)), labels=N_VALUES)
    ax.set_yticks(range(len(C_VALUES)), labels=C_VALUES)
    ax.set_xlabel("Number of nodes n")
    ax.set_ylabel("c")

fig.tight_layout()
if SAVE_OUTPUTS:
    fig.savefig(FIGURES_DIR / "bound_usage_and_approx_success.png", dpi=220, bbox_inches="tight")
plt.show()

## Instance-level audit tables

The first table flags any source Goldberg–Liu result above the requested theoretical bound. It is deliberately not asserted away because the notebook preserves the source IP's threshold convention. The other tables check feasibility.

In [ ]:
bound_violations = df.loc[~df["bound_holds_for_goldberg"]]
goldberg_below_opt = df.loc[~df["goldberg_is_upper_bound"]]
approx_infeasible = df.loc[df["approx_success"] & ~df["approx_feasible"]]

print(f"Goldberg–Liu bound violations: {len(bound_violations)}")
print(f"Goldberg–Liu below exact OPT:  {len(goldberg_below_opt)}")
print(f"Infeasible approx outputs:     {len(approx_infeasible)}")
display(bound_violations)
display(goldberg_below_opt)
display(approx_infeasible)